In [ ]:
from pathlib import Path

import numpy  as np
import pandas as pd
from limosat_analysis import CORE_DISPLACEMENT_COLUMNS, open_limosat
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
from cartopy.feature import LAND

RELEASE_DIR = Path("../data")  # folder with the LiMOSAT release files
PERIOD = "202305_202407"
DATA_PATH = RELEASE_DIR / f"limosat_{PERIOD}.nc"  # one period file of the release
TIME_START = "2024-01-01"
TIME_STOP = "2024-01-04"  # Exclusive UTC boundary

## Read point observations

Install this repository in the notebook environment with `pip install -e ".[notebooks]"` from the repository root. Change `DATA_PATH` and the UTC dates in the first cell.

A row is one observation of a feature in one Sentinel-1 scene. `time` is timezone-aware UTC, `scene` is the Sentinel-1 product name, and `x_m`, `y_m` are EPSG:3413 metres. `trajectory_id` is unique within this period, and the rows of a trajectory are ordered by `point_index`. `primary_only=True` keeps the rows in the period's own date window, so overlapping periods are not double counted.

In [ ]:
data = open_limosat(DATA_PATH)
# By default a read returns the core columns; name the extra ones you need (or columns="all")
df = data.read_points(start=TIME_START, stop=TIME_STOP, primary_only=True,
                      columns=["trajectory_id", "time", "x_m", "y_m", "scene"])
df.head()

## Source information

The date interval includes `TIME_START` and excludes `TIME_STOP`. The attributes record the release id, period, CRS, the flag names and the period's primary window; `data.describe()` lists every column with its type, unit and description.

In [ ]:
df.attrs

In [ ]:
data.describe()

## Plot all points for a given date range

In [ ]:
# Points were already filtered to the configured date range.
df1 = df
n_scenes = df1.scene.nunique()
n_trajectories = df1.trajectory_id.nunique()
print(n_scenes, "scenes in the date range")
print(n_trajectories, "trajectories in the date range")

nps = ccrs.NorthPolarStereo(central_longitude=-45, true_scale_latitude=70.0)
days = (df1.time - df1.time.min()).dt.total_seconds() / 86_400
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, projection=nps)
scat = ax.scatter(df1.x_m, df1.y_m, c=days, cmap="jet", s=1)
plt.colorbar(scat, ax=ax, shrink=0.5, label="days since start")
ax.add_feature(LAND, zorder=10, edgecolor="black", facecolor="lightgray")
plt.show()

In [ ]:
# Calculate the number of points per scene and per trajectory
points_per_scene = df1.groupby("scene").size()
points_per_trajectory = df1.groupby("trajectory_id").size()
fig, axs = plt.subplots(1, 2, figsize=(12, 3))
axs[0].hist(points_per_scene, bins=50)
axs[1].hist(points_per_trajectory, bins=50)
plt.show()

## Plot the longest trajectories

`read_trajectories` returns the complete history of each trajectory, not only the part inside the date range.

In [ ]:
# Trajectories with at least 10 observations in the date range: plot the first 10 of them in full
long_trajectory_ids = points_per_trajectory[points_per_trajectory >= 10].index.values[:10]
tracks = data.read_trajectories(trajectory_ids=long_trajectory_ids)
for trajectory_id, track in tracks.groupby("trajectory_id"):
    plt.plot(track.x_m, track.y_m, "-")

## Plot consecutive ice-drift vectors

`read_displacements` returns one row per displacement between consecutive observations of a trajectory, selected by the time the displacement starts. Vector components are EPSG:3413 metres per second. Both endpoints must lie inside the plotting box.

In [ ]:
xlim = [0, 1e6]
ylim = [-1e6, 0]

moves = data.read_displacements(start=TIME_START, stop=TIME_STOP, primary_only=True,
                                columns=[*CORE_DISPLACEMENT_COLUMNS, "quality_flags"])
in_box = (
    moves.x0_m.between(*xlim) & moves.y0_m.between(*ylim)
    & moves.x1_m.between(*xlim) & moves.y1_m.between(*ylim)
)
vectors = moves.loc[in_box & (moves.elapsed_seconds > 0)].copy()
u = vectors.dx_m / vectors.elapsed_seconds
v = vectors.dy_m / vectors.elapsed_seconds

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, projection=nps)
ax.quiver(vectors.x0_m, vectors.y0_m, u, v, angles="xy", scale_units="xy", scale=0.00001)
ax.add_feature(LAND, zorder=10, edgecolor="black", facecolor="lightgray")
plt.show()

## Quality flags

`quality_flags` is a bit mask on each displacement. The flag names and bits are recorded in the file (`data.metadata["flag_bits"]`). Flags never split trajectories and no displacement is dropped for them: filter on them yourself, for example with `exclude_flags=[...]` in `read_displacements`.

In [ ]:
bits = data.metadata["flag_bits"]
flags = moves["quality_flags"].to_numpy(dtype="int64")
pd.Series({name: int(((flags >> bit) & 1).sum()) for name, bit in bits.items()}, name="displacements with the flag")